# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Date Functions

In [0]:
def parse_mixed_date(df, col_name, output_col=None):
    out_col = output_col if output_col else col_name

    return df.withColumn(
        out_col,
        when(
            trim(col(col_name)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(trim(col(col_name)), "dd-MM-yyyy")
        ).when(
            trim(col(col_name)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(trim(col(col_name)),"dd/MM/yyyy")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy/MM/dd")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}-\d{1,2}-\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy-MM-dd")
        ).otherwise(None)
    )

# Date Cleaning

## Customer Table

In [0]:
customer_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_customer`""")

### Typecasting

In [0]:
customer_df = parse_mixed_date(customer_df, "account_created_date")

In [0]:
customer_df=customer_df.\
            withColumn("is_active",
            when(col("is_active")=="1",True)
            .when(col("is_active")=="0",False)
            .otherwise(None)
            )

In [0]:
customer_df=customer_df.withColumn("customer_id",col("customer_id").cast(IntegerType()))
customer_df=customer_df.withColumn("is_active",col("is_active").cast(BooleanType()))
customer_df=customer_df.withColumn("account_created_date",col("account_created_date").cast(DateType()))

### Trimming Spaces

In [0]:
customer_df=customer_df.withColumn("customer_name",trim(col("customer_name")))

### Creating Surrogate Key

In [0]:
customer_df.createOrReplaceTempView("temp")
customer_key=spark.sql("""
with cte as (
  select distinct customer_name,country,account_created_date from temp
)
select customer_name,country,account_created_date,row_number() over(order by customer_name) as customer_sk from cte 
""")

In [0]:
customer_df=customer_df.join(customer_key,["customer_name","country","account_created_date"],"left")

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {SILVER_SCHEMA_PATH}""")

# Saving Dataframe

## Applying Type 1 SCD for Customer Table

In [0]:
from delta.tables import DeltaTable

target_table_path = f"{SILVER_SCHEMA_PATH}.silver_customer"

if spark.catalog.tableExists(target_table_path):
    target_table = DeltaTable.forName(spark, target_table_path)
    
    target_table.alias("target").merge(
        customer_df.alias("source"),
        "target.customer_id = source.customer_id"  
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    customer_df.write.format("delta").saveAsTable(target_table_path)